In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
project_folder = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

input_path = project_folder / "data" / "selected" / "yrbs_selected_9vars.csv"
df = pd.read_csv(input_path)

print("Input:", input_path)
print("Finns filen?", input_path.exists())
df.head()

Input: c:\Users\emmyk\dev\tdm\week2-cleaning-dataset-yrbs\data\selected\yrbs_selected_9vars.csv
Finns filen? True


,row_id,age,sex,grade,social_media_use,mental_health_not_good,sleep_hours,bmipct,height_orig,weight_orig
0,1,NaN,NaN,NaN,6.0,1.0,3.0,97.08,505,180
1,2,NaN,NaN,NaN,4.0,3.0,5.0,NaN,N N,233
2,3,NaN,NaN,NaN,8.0,2.0,1.0,92.26,506,165
3,4,NaN,NaN,NaN,8.0,3.0,4.0,NaN,N N,105
4,5,NaN,NaN,NaN,6.0,3.0,3.0,7.57,601,125


In [4]:
df_dirty = df.copy()

In [5]:
# 1. Missing values
df_dirty.loc[df_dirty.sample(frac=0.05, random_state=1).index, "age"] = np.nan
df_dirty.loc[df_dirty.sample(frac=0.05, random_state=2).index, "sleep_hours"] = np.nan
df_dirty.loc[df_dirty.sample(frac=0.05, random_state=3).index, "bmipct"] = np.nan

In [6]:
# gör kolumner till object så att vi kan lägga in textfel
df_dirty["age"] = df_dirty["age"].astype("object")
df_dirty["sleep_hours"] = df_dirty["sleep_hours"].astype("object")
df_dirty["bmipct"] = df_dirty["bmipct"].astype("object")
df_dirty["sex"] = df_dirty["sex"].astype("object")
df_dirty["grade"] = df_dirty["grade"].astype("object")
df_dirty["social_media_use"] = df_dirty["social_media_use"].astype("object")
df_dirty["mental_health_not_good"] = df_dirty["mental_health_not_good"].astype("object")

In [7]:
# 2. Text istället för siffror / fel datatyp
df_dirty.loc[df_dirty.sample(10, random_state=4).index, "age"] = "sixteen"
df_dirty.loc[df_dirty.sample(10, random_state=5).index, "sleep_hours"] = "ten"
df_dirty.loc[df_dirty.sample(10, random_state=6).index, "bmipct"] = "high"

In [8]:
# 3. Inkonsekvent text och extra whitespace
df_dirty.loc[df_dirty.sample(10, random_state=7).index, "sex"] = " male"
df_dirty.loc[df_dirty.sample(10, random_state=8).index, "sex"] = "Male "
df_dirty.loc[df_dirty.sample(10, random_state=9).index, "sex"] = "M"

df_dirty.loc[df_dirty.sample(10, random_state=10).index, "mental_health_not_good"] = " always"
df_dirty.loc[df_dirty.sample(10, random_state=11).index, "mental_health_not_good"] = "Always "
df_dirty.loc[df_dirty.sample(10, random_state=12).index, "mental_health_not_good"] = "A"

In [9]:
# 4. Blandade kategorier / fel kategorikodning
df_dirty.loc[df_dirty.sample(10, random_state=13).index, "grade"] = "10th grade"
df_dirty.loc[df_dirty.sample(10, random_state=14).index, "grade"] = "Grade 10"
df_dirty.loc[df_dirty.sample(10, random_state=15).index, "grade"] = 2

df_dirty.loc[df_dirty.sample(10, random_state=16).index, "social_media_use"] = "always online"
df_dirty.loc[df_dirty.sample(10, random_state=17).index, "social_media_use"] = "6"

In [10]:
# 5. Outliers
df_dirty.loc[df_dirty.sample(5, random_state=18).index, "bmipct"] = 999
df_dirty.loc[df_dirty.sample(5, random_state=19).index, "height_orig"] = "999"
df_dirty.loc[df_dirty.sample(5, random_state=20).index, "weight_orig"] = "999"

In [11]:
# 6. Logiska konflikter
# social_media_use = 1 betyder "I do not use social media"
# men vi gör vissa sådana rader extremt höga i mental-health-kategori för att skapa konflikt
idx = df_dirty.sample(10, random_state=21).index
df_dirty.loc[idx, "social_media_use"] = 1
df_dirty.loc[idx, "mental_health_not_good"] = 5

In [12]:
# 7. Dubbletter
df_dirty = pd.concat([df_dirty, df_dirty.sample(15, random_state=22)], ignore_index=True)

In [13]:
# Kontrollera snabbt att fel verkligen finns
print("Missing values:")
print(df_dirty.isnull().sum())

print("\nDubbletter:")
print(df_dirty.duplicated().sum())

print("\nExempel på värden i sex:")
print(df_dirty["sex"].value_counts(dropna=False).head(15))

print("\nExempel på värden i grade:")
print(df_dirty["grade"].value_counts(dropna=False).head(15))

print("\nExempel på värden i social_media_use:")
print(df_dirty["social_media_use"].value_counts(dropna=False).head(15))

Missing values:
row_id                        0
age                       20108
sex                       20088
grade                     20088
social_media_use           4897
mental_health_not_good     4384
sleep_hours                3545
bmipct                     3180
height_orig                 269
weight_orig                 684
dtype: int64

Dubbletter:
15

Exempel på värden i sex:
sex
NaN      20088
M           10
Male        10
 male       10
Name: count, dtype: int64

Exempel på värden i grade:
grade
NaN           20088
Grade 10         10
10th grade       10
2                10
Name: count, dtype: int64

Exempel på värden i social_media_use:
social_media_use
6.0              5881
NaN              4897
8.0              4800
7.0              1183
1.0              1089
5.0               903
4.0               708
2.0               406
3.0               231
always online      10
6                  10
Name: count, dtype: int64


In [14]:
# Spara dirty dataset
output_path = project_folder / "data" / "dirty" / "yrbs_dirty_9vars.csv"
df_dirty.to_csv(output_path, index=False)

print("Sparad dirty fil:", output_path)
print("Finns filen?", output_path.exists())

Sparad dirty fil: c:\Users\emmyk\dev\tdm\week2-cleaning-dataset-yrbs\data\dirty\yrbs_dirty_9vars.csv
Finns filen? True
